# Carvana: understand one retained inventory snapshot

**Question:** which vehicles did the selected searches observe, and is their
coverage complete? Read **query definitions → coverage → source page → vehicle
rows → identity counts**. Normal Run All reads saved results without writes.

The historical example begins with **42 Tesla Model 3 listings from model year
2024**, observed on September 8, 2026. Trace VIN `5YJ3E1ET5RF828714`,
listing `4710782`, and its **$44,990 asking price** through the tables.

| Term used below | What it identifies |
| --- | --- |
| Query / `query_id` | One declared search, such as Tesla / Model 3 / 2024 / ZIP 08542. |
| Page | One response within that search, containing up to 24 listing records. |
| Query attempt / `run_id` | One execution of a query; its report records pages, times and outcome. |
| VIN / `listing_id` | The physical vehicle / Carvana's listing for it. Keep both, and include the retailer in identity keys. |
| Coverage complete | The query's retained pages and unique identities reconcile to its reported total. |

## Settings and table meanings

`PLAN_PATH` selects the query definitions; `RUN_PATH` selects the matching
saved collection report. Change them together to select another historical run.
The whole retained observation interval is used; this lesson has no current-time
cutoff or live collection switch.

`coverage` has one row per query attempt, plus rows for unattempted queries.
`observations` has one row per admitted listing observation in a query attempt.
Overlapping searches can observe the same VIN, so table rows are not automatically
a count of distinct cars. Partial queries can supply observed rows while remaining
incomplete. Read the reported total and status before interpreting an empty table.

Continue through **00 → 10 → 11 → 20 → 24 → 30**; the
[code walkthrough](../docs/code_walkthrough.md) explains how their selected inputs differ.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / "vehicle/src").is_dir():
    ROOT = ROOT / "vehicle"
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src/vehicle_tracker").is_dir():
    raise FileNotFoundError("Open from researchOS, vehicle, or vehicle/notebooks.")
sys.path.insert(0, str(ROOT / "src"))

from vehicle_tracker.carvana import parse_capture
from vehicle_tracker.readiness import query_readiness

PLAN_PATH = ROOT / "config/carvana_search_queries.json"
RUN_PATH = ROOT / "data/experiments/20260908_mvp1000_verified/run_report.json"
plan = json.loads(PLAN_PATH.read_text(encoding="utf-8"))

## 1. Inspect the query before the results

A query is a set of filters. `plan["queries"]` lists those definitions, and the
first table gives each one a `query_id`. The dictionary below it shows the
first-page request for the first query; displaying it does not send it.

The chosen ZIP is delivery/search context, not the vehicle's physical location.
This saved example disables location filtering. Nearby ZIP searches can return
the same cars, so their counts cannot simply be added. These filters define the
sample reviewed here; they do not establish national inventory coverage.


In [2]:
display(pd.DataFrame(plan["queries"])[["query_id", "zip_code", "location_filter", "filters"]])
query = plan["queries"][0]
request_example = dict(filters=query["filters"], pagination=dict(page=1, pageSize=24),
                       sortBy="MostPopular", zip5=query["zip_code"])
if query.get("location_filter", False):
    request_example["requestedFeatures"] = ["LocationBasedPrefiltering"]
print(json.dumps(request_example, indent=2))

,query_id,zip_code,location_filter,filters
0,tesla_model3_2024,08542,False,"{'makes': [{'name': 'Tesla', 'parentModels': [..."
1,tesla_model3_2023,08542,False,"{'makes': [{'name': 'Tesla', 'parentModels': [..."
2,tesla_model3_2022,08542,False,"{'makes': [{'name': 'Tesla', 'parentModels': [..."
3,tesla_model3_2021,08542,False,"{'makes': [{'name': 'Tesla', 'parentModels': [..."
4,tesla_model3_2020,08542,False,"{'makes': [{'name': 'Tesla', 'parentModels': [..."
5,tesla_model3_2025,08542,False,"{'makes': [{'name': 'Tesla', 'parentModels': [..."
6,tesla_model3_2026,08542,False,"{'makes': [{'name': 'Tesla', 'parentModels': [..."
7,toyota_2022,08542,False,"{'makes': [{'name': 'Toyota'}], 'year': {'min'..."


{
  "filters": {
    "makes": [
      {
        "name": "Tesla",
        "parentModels": [
          {
            "name": "Model 3"
          }
        ]
      }
    ],
    "year": {
      "min": 2024,
      "max": 2024
    }
  },
  "pagination": {
    "page": 1,
    "pageSize": 24
  },
  "sortBy": "MostPopular",
  "zip5": "08542"
}


## 2. Verify retained evidence and expose coverage

The collection report points to each query report and saved file. Its SHA-256
hashes are file fingerprints: changed or missing files block their use.
`query_readiness` reopens the retained pages and checks their request context,
timestamps, identities, row counts and claimed completeness.

Read each `coverage` row across:

- `reported_total`: how many results the source said matched this query.
- `admitted_rows`: how many verified listing observations we retained.
- `query_complete` and `reason`: whether the full query reconciled, and why.

A 24-row page from a 42-result query is useful evidence about those 24 listings.
It is not yet the full query. A validated zero-result query, an unattempted query
and a blocked query have different meanings; use their status and reason.
Multiple attempts remain separate, and invalid evidence contributes no observations.


In [3]:
report_paths = []
checkpoint = None
if RUN_PATH.is_file():
    checkpoint = json.loads(RUN_PATH.read_text(encoding="utf-8"))
    for outcome in checkpoint["outcomes"]:
        for filename, expected_hash in outcome.get("artifact_hashes", {}).items():
            artifact = Path(filename)
            if not artifact.is_file() or hashlib.sha256(artifact.read_bytes()).hexdigest() != expected_hash:
                raise ValueError(f"Retained artifact missing or changed: {artifact}")
        if outcome.get("report"):
            report_paths.append(Path(outcome["report"]))
else:
    print("No retained local run. Inventory is unknown, not zero.")

manifest = dict(queries=plan["queries"], population=plan["description"])
coverage, observations = query_readiness(manifest, report_paths)
display(coverage[["query_id", "status", "query_complete", "attempt_versions", "reported_total",
                  "admitted_rows", "reconciliation_difference", "observation_start", "observation_end", "reason"]])
print("All planned queries uniquely complete:", bool(
    coverage.query_complete.all() and coverage.attempt_versions.eq(1).all()))
print('Selected query plan:', PLAN_PATH)
print('Selected saved run:', RUN_PATH)
print('Historical run ended UTC:', checkpoint.get('ended_utc') if checkpoint else 'unavailable')


,query_id,status,query_complete,attempt_versions,reported_total,admitted_rows,reconciliation_difference,observation_start,observation_end,reason
0,tesla_model3_2024,complete_query,True,1,42,42,0,2026-09-08T11:32:58.875154+00:00,2026-09-08T11:33:01.738413+00:00,
1,tesla_model3_2023,complete_query,True,1,196,196,0,2026-09-08T11:33:04.810428+00:00,2026-09-08T11:33:28.667332+00:00,
2,tesla_model3_2022,complete_query,True,1,169,169,0,2026-09-08T11:33:31.770018+00:00,2026-09-08T11:33:52.616274+00:00,
3,tesla_model3_2021,complete_query,True,1,146,146,0,2026-09-08T11:33:55.777053+00:00,2026-09-08T11:34:13.640376+00:00,
4,tesla_model3_2020,complete_query,True,1,105,105,0,2026-09-08T11:34:16.863185+00:00,2026-09-08T11:34:28.665343+00:00,
5,tesla_model3_2025,complete_query,True,1,27,27,0,2026-09-08T11:34:31.813480+00:00,2026-09-08T11:34:34.647712+00:00,
6,tesla_model3_2026,complete_query,True,1,9,9,0,2026-09-08T11:34:37.788042+00:00,2026-09-08T11:34:37.788042+00:00,
7,toyota_2022,partial,False,1,529,312,-217,2026-09-08T11:34:40.762836+00:00,2026-09-08T11:35:16.865121+00:00,Target reached; remaining query pages were not...


All planned queries uniquely complete: False
Selected query plan: c:\Users\Sean\VscProjects\researchOS\vehicle\config\carvana_search_queries.json
Selected saved run: c:\Users\Sean\VscProjects\researchOS\vehicle\data\experiments\20260908_mvp1000_verified\run_report.json
Historical run ended UTC: 2026-09-08T11:35:16.892452+00:00


## 3. Trace one saved page into normalized columns

Start with the first row of `observations`. Its `report_path` identifies the
query report; its `capture_id` identifies the retained page within that report.
The code opens that page's `retained_source` path and parses its vehicles again.
The displayed first five rows are a preview of that page, not a new sample or
another collection.

| Source field | Analysis column | Meaning / unit |
| --- | --- | --- |
| `vehicleId` | `listing_id` | Carvana listing identifier, stored as text |
| `vin` | `vin` | Physical vehicle identifier within retailer |
| `year`, `make`, `model` | Same names | Published model year and labels |
| `mileage` | `mileage_miles` | Published odometer, miles |
| `price.total` | `asking_price_usd` | Advertised asking price in USD |
| `transportCost` | `transport_cost_usd` | Separate native transport charge in USD |
| `isPurchasePending` | `purchase_pending` | Source's purchase-pending flag; not a completed sale |
| `vehicleLockType` | `vehicle_lock_type` | Native code; no invented sale mapping |
| Capture timestamp | `observed_at_utc` | When the source was observed, in UTC |

`read_query_evidence` adds transport, pending and lock columns to
`observations`; the base parser also preserves native status in `card_text`.
`availability_native` is missing because this source does not provide that older
schema label. Unknown values stay missing; they do not become zero or false.


In [4]:
if not observations.empty:
    first = observations.iloc[0]
    report = json.loads(Path(first.report_path).read_text(encoding="utf-8"))
    page = next(p for p in report["pages"] if p["source_sha256"] == first.capture_id)
    source_path = Path(page["retained_source"])
    source = json.loads(source_path.read_text(encoding="utf-8"))
    print("Retained source:", source_path)
    print("Observed UTC:", source["captured_at_utc"])
    display(pd.DataFrame([source["pagination"]]))
    display(pd.json_normalize(source["vehicles"]).head(5))
    parsed = parse_capture(source)
    display(parsed[["listing_id", "vin", "year", "make", "model", "mileage_miles",
                    "asking_price_usd", "availability_native", "card_text"]].head(5))
else:
    print("No admitted observations to trace. Inspect coverage above.")

Retained source: C:\Users\Sean\VscProjects\researchOS\vehicle\data\experiments\20260908_mvp1000_verified\tesla_model3_2024\raw\b0db523034e4a8079e8db985890ca99fcf3bda4475425a97ba2ce126306b6347.json
Observed UTC: 2026-09-08T11:32:58.875154+00:00


,currentPage,pageSize,totalMatchedInventory,totalMatchedPages
0,1,24,42,2


,isOnDemand,isPurchasePending,make,mileage,model,parentModel,transportCost,vehicleId,vehicleInventoryType,vehicleLockType,vehiclePurchaseType,vin,year,price.total
0,False,False,Tesla,17120,Model 3,Model 3,190.0,4710782,1,0,Purchasable,5YJ3E1ET5RF828714,2024,44990.0
1,False,False,Tesla,49843,Model 3,Model 3,290.0,4685821,1,0,Purchasable,5YJ3E1EA1RF730116,2024,32990.0
2,False,False,Tesla,24433,Model 3,Model 3,1590.0,4453081,1,0,Purchasable,5YJ3E1ET9RF827954,2024,42990.0
3,False,False,Tesla,14270,Model 3,Model 3,1890.0,4718862,1,0,Purchasable,5YJ3E1ET0RF855996,2024,45990.0
4,False,False,Tesla,60011,Model 3,Model 3,1890.0,4715298,1,0,Purchasable,5YJ3E1EA5RF732791,2024,31990.0


,listing_id,vin,year,make,model,mileage_miles,asking_price_usd,availability_native,card_text
0,4710782,5YJ3E1ET5RF828714,2024,Tesla,Model 3,17120,44990.0,None,"{""isOnDemand"": false, ""isPurchasePending"": fal..."
1,4685821,5YJ3E1EA1RF730116,2024,Tesla,Model 3,49843,32990.0,None,"{""isOnDemand"": false, ""isPurchasePending"": fal..."
2,4453081,5YJ3E1ET9RF827954,2024,Tesla,Model 3,24433,42990.0,None,"{""isOnDemand"": false, ""isPurchasePending"": fal..."
3,4718862,5YJ3E1ET0RF855996,2024,Tesla,Model 3,14270,45990.0,None,"{""isOnDemand"": false, ""isPurchasePending"": fal..."
4,4715298,5YJ3E1EA5RF732791,2024,Tesla,Model 3,60011,31990.0,None,"{""isOnDemand"": false, ""isPurchasePending"": fal..."


## 4. Check identities and missing values before using counts

Read these tables as checks on what can be counted:

1. `isna().sum()` counts unknown values without filling them.
2. The query groups count rows, listing IDs and VINs separately.
3. Duplicate rows show overlapping listing memberships across the selected queries.
4. The two identity checks expose one listing ID tied to different VINs, or one VIN
   tied to different listing IDs.

The final union counts distinct `(retailer, listing_id)` pairs in this sample.
It does not choose a preferred price among duplicate observations or merge listing
histories. All source rows stay in `observations`; a clean identity check does
not by itself prove complete query coverage or a sale.


In [5]:
if not observations.empty:
    display(observations[["vin", "year", "mileage_miles", "asking_price_usd",
                          "purchase_pending", "vehicle_lock_type", "transport_cost_usd"]]
            .isna().sum().rename("missing_values").to_frame())
    display(observations.groupby("query_id").agg(
        observations=("listing_id", "size"), unique_listings=("listing_id", "nunique"),
        unique_vins=("vin", "nunique")))
    duplicates = observations[observations.duplicated(["retailer", "listing_id"], keep=False)]
    display(duplicates[["query_id", "retailer", "listing_id", "vin", "observed_at_utc", "report_path"]])
    listing_vins = observations.groupby(["retailer", "listing_id"]).vin.nunique()
    vin_listings = observations.groupby(["retailer", "vin"]).listing_id.nunique()
    display(listing_vins[listing_vins.gt(1)].rename("conflicting_vins").to_frame())
    display(vin_listings[vin_listings.gt(1)].rename("listing_ids_for_vin").to_frame())
    print("Observed sample union:", len(observations[["retailer", "listing_id"]].drop_duplicates()))
    print("Capture interval UTC:", observations.observed_at_utc.min(), "to", observations.observed_at_utc.max())
    print("National inventory coverage: UNVERIFIED. No sales count is produced.")

,missing_values
vin,0
year,0
mileage_miles,0
asking_price_usd,0
purchase_pending,0
vehicle_lock_type,0
transport_cost_usd,20


,observations,unique_listings,unique_vins
query_id,,,
tesla_model3_2020,105,105,105
tesla_model3_2021,146,146,146
tesla_model3_2022,169,169,169
tesla_model3_2023,196,196,196
tesla_model3_2024,42,42,42
tesla_model3_2025,27,27,27
tesla_model3_2026,9,9,9
toyota_2022,312,312,312


,query_id,retailer,listing_id,vin,observed_at_utc,report_path


,,conflicting_vins
retailer,listing_id,


,,listing_ids_for_vin
retailer,vin,


Observed sample union: 1006
Capture interval UTC: 2026-09-08T11:32:58.875154+00:00 to 2026-09-08T11:35:16.865121+00:00
National inventory coverage: UNVERIFIED. No sales count is produced.


### Try it: a row is not the whole query

Find VIN `5YJ3E1ET5RF828714` in the displayed source and normalized rows. Verify
its listing ID **4710782** and asking price **44990**. Then inspect the first
query's `reported_total` and `admitted_rows`: the retained example has **42**
matching listings across two pages, even though the first page has **24** rows.
Use `observations.loc[observations.vin.eq('5YJ3E1ET5RF828714')]` to inspect the row.
An empty match means this VIN is not in the selected evidence; it does not mean a sale.

In Notebook 11, changing make/model/year/ZIP changes a request preview. Here,
changing the file paths changes the evidence you read. Neither action alone collects data.


## What to conclude from this snapshot

`coverage` establishes which declared queries completed and when. `observations`
contains their admitted rows; the identity and missing-field tables explain what
can be counted. A successful query covers that search during its capture window,
not national inventory. Asking prices and native pending flags remain observations.

Continue to [Notebook 11](11_carvana_live_collection_lab.ipynb) for the explicit one-page learning lab,
then [Notebook 20](20_carvana_history_analysis.ipynb) for daily collection
health, comparable VIN changes and matched prices. The [daily-cycle guide](../docs/daily_cycles.md)
explains collection windows and recovery; [the historical collection evaluation](../docs/collection_evaluation_20260908.md)
retains earlier method comparisons.

### Optional implementation references

[history.py](../src/vehicle_tracker/history.py) and [readiness.py](../src/vehicle_tracker/readiness.py)
read and validate the evidence used above. [carvana.py](../src/vehicle_tracker/carvana.py)
parses fields. The separate collection path is [collect_carvana_search.py](../scripts/collect_carvana_search.py),
[search_plan.py](../src/vehicle_tracker/search_plan.py), [search.py](../src/vehicle_tracker/search.py)
and [storage.py](../src/vehicle_tracker/storage.py). The historical walkthrough below
explains that path; its snippets are documentation, not executable notebook cells.


## Optional historical walkthrough: how the 42 Tesla listings were collected

Think of selecting **Tesla -> Model 3 -> 2024** on a car-search website and clicking through its results. Our Python program sends the search instructions directly to the server that supplies those results. It does not open 42 separate vehicle webpages.

### 1. Python asks for the first page

This is the address receiving every search request:

```text
https://apik.carvana.io/merch/search/api/v2/search
```

The URL stays the same. The `request` dictionary tells it which vehicles and which page we want:

```python
request = {
    "filters": {
        "makes": [{"name": "Tesla", "parentModels": [{"name": "Model 3"}]}],
        "year": {"min": 2024, "max": 2024}
    },
    "pagination": {"page": 1, "pageSize": 24},
    "sortBy": "MostPopular",
    "zip5": "08542"
}

response = requests.post(ENDPOINT, json=request, ...)
data = response.json()
```

Read the first line as **"send this search to Carvana."** Read the second as **"turn Carvana's answer into a Python dictionary I can work with."** The `...` abbreviates timeout and header settings. These snippets explain the running collector; they are Markdown, not executable notebook cells.

### 2. Carvana's answer already contains the vehicle data

In the retained September 8, 2026 example, page 1 reported **42 matching vehicles over two pages** and contained **24 vehicle records**. The shortened structure below reconstructs selected saved fields; it is not the complete original HTTP response:

```python
data = {
    "inventory": {
        "pagination": {
            "currentPage": 1,
            "pageSize": 24,
            "totalMatchedInventory": 42,
            "totalMatchedPages": 2
        },
        "vehicles": [
            {
                "vehicleId": 4710782,
                "vin": "5YJ3E1ET5RF828714",
                "year": 2024,
                "make": "Tesla",
                "model": "Model 3",
                "mileage": 17120,
                "price": {"total": 44990.0}
            }
            # Another 23 vehicle records were returned on this page.
        ]
    }
}
```

**There is no second scrape needed to obtain these prices or VINs: they are already inside the POST response.** For example:

```python
vehicles = data["inventory"]["vehicles"]  # the list of 24 vehicle records
first_vehicle = vehicles[0]              # one record from that list
first_vehicle["price"]["total"]          # 44990.0
```

### 3. Python turns those records into table rows

Each vehicle dictionary becomes one row. Our parser gives the source fields consistent column names:

| Source field in the answer | Column in our table | First saved vehicle |
| --- | --- | --- |
| `vehicleId` | `listing_id` | 4710782 |
| `vin` | `vin` | 5YJ3E1ET5RF828714 |
| `year` | `year` | 2024 |
| `mileage` | `mileage_miles` | 17120 |
| `price.total` | `asking_price_usd` | 44990.0 |

The actual collector performs this transformation with:

```python
capture = project_response(data, request, observed_at=capture_time)
frame = parse_search_capture(capture)
```

`project_response` keeps the selected source fields and attaches the request and capture time. `parse_search_capture` checks the fields and builds a pandas DataFrame: **24 rows for this page**. The price is an asking price in USD; it is not a transaction price.

### 4. Python saves the page so it survives after the script ends

The response and DataFrame are initially only in memory. The collector writes two useful forms to your computer:

```python
retained = retain_capture(capture, destination / "raw")
store_capture(
    destination / "vehicle.sqlite",
    run_id=run_id,
    page_number=1,
    raw_file=retained,
)
```

- The JSON file under `raw/` preserves selected source fields, the request and observation time. It lets us check where a value came from later. It is not the entire original HTTP body.
- `vehicle.sqlite` is a local database file containing the vehicle rows and a record of the page attempt. `store_capture` reads and parses the saved JSON before inserting its rows.

In the actual implementation, the JSON is retained before parsing/coverage checks so failed evidence can also be kept. Vehicle rows are admitted to the database only after the checks pass; failed attempts are recorded separately.

### 5. Python requests page 2 and adds its 18 vehicles

The answer told us there were two pages. The collector waits according to its request pacing and repeats the same process with:

```python
request["pagination"]["page"] = 2
```

The URL, Tesla filters and ZIP stay the same. Only the requested page number changes.

| Request | Vehicles returned and saved | Running count of distinct listings |
| --- | ---: | ---: |
| Page 1 | 24 | 24 |
| Page 2 | 18 | 42 |

Both counts above come from the saved trial. The collector checks for repeated identities, changed totals and invalid data before adding a page. At the end, **42 distinct saved listings matched the source's 42-result total**, so this particular query was marked complete.

`run_report.json` records the pages, counts, times, file hashes and whether the query completed. Getting HTTP 200 alone would not establish completeness.

### 6. The collector repeats this for the next search

The plan file is just a list of searches, for example 2024 Model 3, then 2023 Model 3, then 2022 Model 3. `collect_plan` goes through that list; `collect_search` handles the pages within each search.

```text
Chosen search plan
    |
    +-- 2024 Tesla Model 3
    |       POST page 1 -> check and save 24 rows
    |       POST page 2 -> check and save 18 rows -> query complete
    |
    +-- 2023 Tesla Model 3
    |       POST page 1 -> check and save rows
    |       POST page 2 -> ... until that search finishes or stops
    |
    +-- next chosen search ...
```

**That repeated request/check/save process is the collection.** A sample target or request/time limit can stop the process early. Earlier saved rows remain useful observations, but unfinished searches remain incomplete.

### 7. The notebook reads what the collector already saved

The collector performs the live requests and file/database writes. **This notebook is the explanation and inspection step afterward.** Earlier in this notebook, `query_readiness` reads the saved reports and JSON pages, checks their evidence, and rebuilds the `observations` DataFrame without another POST. That is why Run All can show vehicles while remaining offline.

For this example, the evidence is in [the saved 2024 Tesla query report](../data/experiments/20260908_mvp1000_verified/tesla_model3_2024/run_report.json), with its linked source files. These are historical observations, not today's inventory. A new day's collection must make fresh requests and use a new run destination.

A complete Tesla query is still only that selected search. The ZIP is shopper context, this request omits location filtering, and neither the 42-result reconciliation nor adding more searches proves complete national coverage. Missing listings are not confirmed sales.